# Auto correct OCR

## adding synonyms column for future

In [2]:
import pandas as pd

# Load your existing cleaned ingredient database
df = pd.read_csv('paulaschoice_us_ingredients_clean.csv')

# Add a blank Synonyms column if it doesn't already exist
if 'Synonyms' not in df.columns:
    df['Synonyms'] = ""

# Save updated database
df.to_csv('paulaschoice_us_ingredients_clean.csv', index=False)

print("✅ Synonyms column added (blank for now). Updated file saved.")

✅ Synonyms column added (blank for now). Updated file saved.


In [3]:
df

,Name,Description,Rating,Synonyms
0,Cranberry Seed Extract,Natural ingredient derived from antioxidant/om...,average,NaN
1,Crataegus Monogyna Extract,"Also known as hawthorn berry, this plant extra...",average,NaN
2,D&C,"According to the FDA, D&C is an identification...",average,NaN
3,D&C Colors,General term for a wide group of color additiv...,average,NaN
4,Daucus Carota,Also known as wild carrot. An antioxidant-rich...,average,NaN
...,...,...,...,...
26013,Zostera Marina Extract,Functions: Skin Conditioning. We have not yet ...,not rated,NaN
26014,Zygophyllum Qatarense Leaf/Stem Extract,Functions: Antioxidant. We have not yet rated ...,not rated,NaN
26015,Zygosaccharomyces Microellipsoides Ferment,Functions: Humectant. We have not yet rated th...,not rated,NaN
26016,Zymomonas Ferment Extract,"Functions: Humectant, Skin Conditioning. We ha...",not rated,NaN


## bulk processor helper with ocr autocorrect before matching 

In [5]:
import pandas as pd
from rapidfuzz import process, fuzz

# Load your ingredient database
# Assumes columns: ['Name', 'Description', 'Rating']
db = pd.read_csv('paulaschoice_us_ingredients_clean.csv')

# Build known terms list from 'Name' column (synonyms paused for now)
known_terms = db['Name'].str.lower().tolist()


def clean_text(text):
    """Basic text cleaning for OCR outputs."""
    return text.lower().strip()


def auto_correct_ocr(ocr_text, threshold=80):
    """Attempt to correct OCR string using known ingredient names."""
    cleaned = clean_text(ocr_text)
    match, score, _ = process.extractOne(cleaned, known_terms, scorer=fuzz.token_sort_ratio)
    if score >= threshold:
        return match  # Corrected string
    return cleaned  # Return as-is if no good match


def match_ingredient(ingredient, threshold=80):
    """Fuzzy match corrected ingredient to database."""
    corrected = auto_correct_ocr(ingredient, threshold)
    match, score, _ = process.extractOne(corrected, known_terms, scorer=fuzz.token_sort_ratio)
    if score >= threshold:
        db_row = db[db['Name'].str.lower() == match].iloc[0]
        return {
            'Input': ingredient,
            'Corrected': corrected,
            'Matched Name': db_row['Name'],
            'Description': db_row['Description'],
            'Rating': db_row['Rating'],
            'Score': score
        }
    else:
        return {
            'Input': ingredient,
            'Corrected': corrected,
            'Matched Name': None,
            'Description': None,
            'Rating': None,
            'Score': score
        }


def bulk_process(ingredient_list, threshold=80, unmatched_log='unmatched_ingredients.csv'):
    """Process list of OCR outputs, apply correction & matching."""
    results = []
    unmatched = []

    for ing in ingredient_list:
        res = match_ingredient(ing, threshold)
        results.append(res)
        if res['Matched Name'] is None:
            unmatched.append(res)

    df_results = pd.DataFrame(results)
    if unmatched:
        pd.DataFrame(unmatched).to_csv(unmatched_log, index=False)
        print(f"⚠️ Logged {len(unmatched)} unmatched ingredients to '{unmatched_log}'")
    return df_results


# Example usage:
ocr_outputs = ["ethylhexglcerin", "sodium hualronate", "xyzrandomchemical"]
df_results = bulk_process(ocr_outputs)
print(df_results)

⚠️ Logged 1 unmatched ingredients to 'unmatched_ingredients.csv'
               Input           Corrected        Matched Name  \
0    ethylhexglcerin  ethylhexylglycerin  Ethylhexylglycerin   
1  sodium hualronate  sodium hyaluronate  Sodium Hyaluronate   
2  xyzrandomchemical   xyzrandomchemical                None   

                                         Description Rating       Score  
0  Skin-softening agent with hydration boosting p...   good  100.000000  
1  The more bio-available, salt form of hyaluroni...   best  100.000000  
2                                               None   None   55.555556  


## OCR Pipeline

In [9]:
import pytesseract
from PIL import Image
from rapidfuzz import process, fuzz
import pandas as pd
import re

# ------------------ Setup (Assumes db is already loaded) ------------------
# db = pd.read_csv("paulaschoice_us_ingredients_clean.csv")  # Uncomment if needed

known_terms = set(db['Name'].str.lower())
for synonyms in db.get('Synonyms', pd.Series()).dropna():
    for syn in str(synonyms).split(','):
        known_terms.add(syn.strip().lower())
known_terms = list(known_terms)

# ------------------ Helper Functions ------------------

def clean_text(text):
    return re.sub(r'[^a-zA-Z0-9 ]', '', text).lower().strip()

def auto_correct_ocr(ocr_text, threshold=80):
    cleaned = clean_text(ocr_text)
    match = process.extractOne(cleaned, known_terms, scorer=fuzz.token_sort_ratio)
    if match and match[1] >= threshold:
        return match[0]
    return cleaned

def match_ingredient(ingredient, threshold=80):
    corrected = auto_correct_ocr(ingredient)
    match = process.extractOne(corrected, db['Name'], scorer=fuzz.token_sort_ratio)
    if match and match[1] >= threshold:
        db_row = db[db['Name'] == match[0]].iloc[0]
        return {
            "Input": ingredient,
            "Corrected": corrected,
            "Matched Name": db_row['Name'],
            "Description": db_row['Description'],
            "Rating": db_row['Rating'],
            "Score": match[1]
        }
    else:
        return {
            "Input": ingredient,
            "Corrected": corrected,
            "Matched Name": None,
            "Description": None,
            "Rating": None,
            "Score": match[1] if match else 0
        }

def process_image(image_path, threshold=80):
    """Full pipeline: Image -> Text -> Matching"""
    img = Image.open(image_path)
    ocr_output = pytesseract.image_to_string(img)
    print("\n🔎 OCR Raw Output:\n", ocr_output)

    # Basic cleaning - split by lines, remove empty lines
    ingredients = [line.strip() for line in ocr_output.split('\n') if line.strip()]
    print("\n🧾 Extracted Ingredient Lines:", ingredients)

    if not ingredients:
        print("⚠️ No ingredients detected from image.")
        return pd.DataFrame()  # Empty result

    # Process each ingredient line
    results = [match_ingredient(ing, threshold) for ing in ingredients]
    df_results = pd.DataFrame(results)

    if not df_results.empty and 'Matched Name' in df_results.columns:
        unmatched = df_results[df_results['Matched Name'].isna()]
        if not unmatched.empty:
            unmatched.to_csv("unmatched_ingredients.csv", mode='a', index=False, header=False)
            print(f"⚠️ Logged {len(unmatched)} unmatched ingredients to 'unmatched_ingredients.csv'")

    return df_results


In [11]:
# ------------------ Example ------------------
df = process_image("label1.png")
print(df)


🔎 OCR Raw Output:
 INGREDIENTS: Aqua, Cyclopentasiloxane, Glycerine, Caprylic/-
Capric Triglyceride, Dimethicone, Isohexadecane, Cetyl Alcohol,
Glyceryl Stearal (and) PEG-100 Stearate, Butyrospermum

ter, Xylitylglucoside [and] Anhydroxylitol (and)
ide, Glycerin (and) Aqua [and] Butylene Glycol

a nd) Polysorbate 20 (and) Palmitoyl

) Palmitoyl Tetrapeptide-7, Fragaria Ananassa
rou it Extract, Sodium Hyaluronate, Rubus Chamae-
morus ) Seed Oil, Panthenol, Cetearyl Alcohol, Poly-
xane, Polysorbate 80, Ammonium Acryloyl-

VP Copolymer, Acrylates/C 10-30 Alkyl Acrylate
Wlantoin, Disodium EDTA, Tocopheryl Acetate,
e, Phenoxyethanol (and) Ethythexylglycerin.



🧾 Extracted Ingredient Lines: ['INGREDIENTS: Aqua, Cyclopentasiloxane, Glycerine, Caprylic/-', 'Capric Triglyceride, Dimethicone, Isohexadecane, Cetyl Alcohol,', 'Glyceryl Stearal (and) PEG-100 Stearate, Butyrospermum', 'ter, Xylitylglucoside [and] Anhydroxylitol (and)', 'ide, Glycerin (and) Aqua [and] Butylene Glycol', 'a nd) Polys

In [14]:
import cv2
import numpy as np
from PIL import Image
import pytesseract
import pandas as pd
from rapidfuzz import process, fuzz

def preprocess_image(image_path):
    """Preprocess image for better OCR."""
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
    return thresh


def extract_ingredients(text):
    """Parse OCR text to clean list of ingredient strings."""
    lines = text.split(',')  # Split on commas
    cleaned = []
    for line in lines:
        line = line.lower().strip()
        line = line.replace('(and)', '').replace('[and]', '')
        line = line.replace('-', '').replace('ingredients:', '')
        line = line.strip()
        if line:
            cleaned.append(line)
    return cleaned


def auto_correct_ocr(ocr_text, threshold=80):
    """Correct ingredient using known_terms."""
    match, score, _ = process.extractOne(ocr_text, known_terms, scorer=fuzz.token_sort_ratio)
    if score >= threshold:
        return match
    return ocr_text


def match_ingredient(ingredient, threshold=80):
    """Fuzzy match corrected ingredient to database."""
    corrected = auto_correct_ocr(ingredient, threshold)
    match, score, _ = process.extractOne(corrected, db['Name'], scorer=fuzz.token_sort_ratio)
    if score >= threshold:
        row = db[db['Name'] == match].iloc[0]
        return {
            'Input': ingredient,
            'Corrected': corrected,
            'Matched Name': row['Name'],
            'Description': row['Description'],
            'Rating': row['Rating'],
            'Score': score
        }
    return {
        'Input': ingredient,
        'Corrected': corrected,
        'Matched Name': None,
        'Description': None,
        'Rating': None,
        'Score': score
    }


def process_image(image_path, threshold=80):
    """Full pipeline: Image -> Text -> Ingredient Matching."""
    img = preprocess_image(image_path)
    ocr_text = pytesseract.image_to_string(img)
    ingredients = extract_ingredients(ocr_text)

    results = [match_ingredient(ing, threshold) for ing in ingredients]
    df_results = pd.DataFrame(results)

    unmatched = df_results[df_results['Matched Name'].isna()]
    if not unmatched.empty:
        unmatched.to_csv("unmatched_ingredients.csv", mode='a', index=False, header=False)
        print(f"⚠️ Logged {len(unmatched)} unmatched ingredients to 'unmatched_ingredients.csv'")

    return df_results

# ------------------ Example ------------------
df = process_image("label1.png")
print(df)

⚠️ Logged 9 unmatched ingredients to 'unmatched_ingredients.csv'
                                                Input  \
0                                     noredients agua   
1                                  cyclopentasiloxane   
2                                           glycerine   
3                                  caprylic/\n| pride   
4                                       de methic one   
5                                       isohexadecane   
6                                       cetyl alcohol   
7                         rate (and! peg 109 stearate   
8                            butyrospermum\n3) butter   
9   \wwiitvlgt hicowde (and] anhydroxylitol \ning ide   
10  ciye erin  aqua  butylene glycol\nnel ‘land’ p...   
11                fragaria ananassa\ntier it ext) act   
12                                 sodium hyaluronate   
13                      rubus cnamae\ngerry] gord oil   
14                                          panthenol   
15           cetearyl a